# Mise en place de la VM Monitoring & Analytics

Ce notebook crée la **VM EC2** de l'architecture (`doc/G10-architecture-V2.md`) : le **Scenario Analyzer** et **Grafana OSS** y tournent dans Docker, avec le même `AWS/Grafana/docker-compose.local.yml` qu'en local. Sur la VM, l'Analyzer lit la base **Amazon Timestream** (`SdatawayIoTDB.telemetry`) alimentée par la Lambda Ingestor. Le **rejeu de fichiers** (`batches.jsonl`) reste disponible (`ANALYSER_SOURCE = "file"`, étape 8).

Grafana est publié sur l'**adresse IP publique** de la VM, `http://<ip-publique>:3000`. Le security group n'ouvre que ce port, et seulement aux adresses de `GRAFANA_CIDRS` (étape 0) : par défaut, l'IP publique de ce poste. Il n'y a ni SSH ni clé : l'administration passe par **SSM**, et un tunnel SSM reste possible (étape 9).

Chaque cellule peut être relancée sans risque. L'arrêt (10) et le nettoyage (11) sont désactivés par défaut.

**Prérequis** : identifiants AWS configurés (`aws configure`) avec les droits EC2, IAM, S3 et SSM ; jeu de rejeu généré (voir `AWS/Analyser/README.md`), toujours inclus dans l'archive ; pour le tunnel (facultatif), l'AWS CLI et le plugin Session Manager (`brew install awscli && brew install --cask session-manager-plugin`).

## 0. Configuration

À adapter si besoin. `PREFIX` sert à nommer toutes les ressources : le rôle, le security group, l'instance et le bucket, auquel s'ajoute l'identifiant du compte.

`GRAFANA_CIDRS` liste les adresses autorisées à joindre Grafana sur son IP publique :
- `[resources.caller_public_cidr()]` (défaut) : l'IP publique de ce poste seulement, lue sur `checkip.amazonaws.com` ;
- `["0.0.0.0/0"]` : tout Internet, par exemple pour une démo. Seul le mot de passe administrateur protège alors Grafana ;
- `[]` : aucun port ouvert, accès par le tunnel SSM uniquement.

Grafana est servi en **HTTP**, sans TLS : le mot de passe circule en clair. D'où le défaut restreint à ce poste. Changer la liste, puis relancer les étapes 4 et 8, applique le changement.

In [2]:
import json
import shutil
import sys
from pathlib import Path

import boto3

try:
    import vm_setup  # noqa: F401
except ModuleNotFoundError:  # iCloud peut masquer le .pth de l'installation éditable
    sys.path.insert(0, str(Path.cwd() / "src"))
from vm_setup import bundle, resources, scripts
from vm_setup.resources import Clients, VmConfig

REGION = "us-east-1"        # même région que AWS/IotCore/setup_iot_core.ipynb
INSTANCE_TYPE = "t3.small"  # 2 Go : Analyzer (rejeu en mémoire) + Grafana + build Docker
PREFIX = "coffee-vm"
GRAFANA_CIDRS = ["0.0.0.0/0"] #[resources.caller_public_cidr()]  # ["0.0.0.0/0"] : tout Internet ; [] : tunnel SSM seulement

ROOT = Path.cwd().resolve().parents[1]   # racine du dépôt (le notebook est dans AWS/VM/)
STATE_FILE = Path.cwd() / ".state.json"  # identifiants des ressources, ignoré par git
assert (ROOT / "AWS/Grafana/docker-compose.local.yml").exists(), "lancer le notebook depuis AWS/VM/"

session = boto3.Session(region_name=REGION)
clients = Clients.from_session(session)
ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]
cfg = VmConfig(
    region=REGION, account_id=ACCOUNT_ID, prefix=PREFIX, instance_type=INSTANCE_TYPE,
    grafana_cidrs=tuple(GRAFANA_CIDRS),
)
print("compte", ACCOUNT_ID, "| région", REGION, "| dépôt", ROOT)
print("Grafana public pour :", ", ".join(cfg.grafana_cidrs) or "personne (tunnel SSM seulement)")


def load_state():
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}


def save_state(**values):
    state = load_state() | values
    STATE_FILE.write_text(json.dumps(state, indent=2))
    return state

compte 352909266144 | région us-east-1 | dépôt /home/gasile/Documents/MASTER/CloudSys/projet/repo/cloud_project_2_grp10
Grafana public pour : 0.0.0.0/0


## 1. Prérequis

On vérifie, sans rien créer :
- qu'un VPC par défaut existe, car la VM s'y place et reçoit une IP publique, qui sert à Grafana et à joindre SSM, Docker Hub et GitHub en sortie ;
- que les fichiers à déployer sont présents, dont le jeu de rejeu ;
- que le plugin Session Manager est installé en local (il ne sert qu'au tunnel facultatif de l'étape 9).

In [3]:
vpc_id = resources.default_vpc_id(clients.ec2)
files = bundle.bundle_files(ROOT)
plugin = shutil.which("session-manager-plugin")
print("VPC par défaut :", vpc_id)
print("fichiers à déployer :", len(files))
print("plugin Session Manager :", plugin or "absent (tunnel facultatif) : brew install --cask session-manager-plugin")

VPC par défaut : vpc-07f1d9fb1fef102bf
fichiers à déployer : 23
plugin Session Manager : absent (tunnel facultatif) : brew install --cask session-manager-plugin


## 2. Mot de passe administrateur de Grafana

Le mot de passe est généré une seule fois et stocké dans **SSM Parameter Store** (`SecureString`, chiffré, niveau standard gratuit). Il n'apparaît jamais dans ce notebook : la VM le lit elle-même au déploiement. Pour se connecter à Grafana :

```bash
aws ssm get-parameter --name /coffee-vm/grafana-admin-password --with-decryption --query Parameter.Value --output text --region us-east-1
```

(Le nom suit `PREFIX` : `/<PREFIX>/grafana-admin-password`.)

Grafana enregistre ce mot de passe à son **premier** démarrage, dans son volume. Le modifier ensuite dans Parameter Store n'a aucun effet (voir le README).

In [4]:
created = resources.ensure_admin_password(clients.ssm, cfg)
print("paramètre", "créé" if created else "existant", ":", cfg.parameter_name)

paramètre existant : /coffee-vm/grafana-admin-password


## 3. Rôle IAM de la VM

La VM reçoit un rôle, via un *instance profile*, réduit au strict nécessaire :
- `AmazonSSMManagedInstanceCore` : l'agent SSM (Run Command et tunnel) ;
- lecture de **la seule** archive de déploiement dans S3 ;
- lecture du **seul** paramètre du mot de passe Grafana ;
- `timestream:Select` sur **la seule** table `SdatawayIoTDB.telemetry`, plus `timestream:DescribeEndpoints`, que le SDK appelle avant toute requête et qui ne peut pas être limité à une ressource.

Relancer cette cellule met à jour les droits d'un rôle existant.

Le conteneur de l'Analyzer utilise ce rôle pour lire Timestream : IMDSv2 accepte donc **deux** sauts réseau (étape 6). Le conteneur Grafana peut aussi l'obtenir. On l'accepte, car le rôle ne lit rien que Grafana n'ait déjà (son mot de passe, la télémétrie qu'il affiche) ou qui soit secret (l'archive).

In [5]:
profile_arn = resources.ensure_instance_role(clients.iam, cfg)
save_state(instance_profile_arn=profile_arn)
print("instance profile :", profile_arn)

instance profile : arn:aws:iam::352909266144:instance-profile/coffee-vm-role


## 4. Security group

**Une seule règle entrante** : le port de Grafana (TCP 3000), depuis les adresses de `GRAFANA_CIDRS`. Ni SSH ni autre port. Avec `GRAFANA_CIDRS = []`, aucune règle entrante. Toute autre règle, ajoutée à la main par exemple, est retirée. Toutes les sorties restent autorisées, pour l'agent SSM, les images Docker et le plugin Grafana Infinity.

In [6]:
sg_id = resources.ensure_security_group(clients.ec2, cfg, vpc_id)
save_state(security_group_id=sg_id)
print("security group :", sg_id, "| port 3000 ouvert à :", ", ".join(cfg.grafana_cidrs) or "personne")

security group : sg-028cfda1fe92eee66 | port 3000 ouvert à : 0.0.0.0/0


## 5. Archive de déploiement

Une liste blanche de fichiers est envoyée dans un bucket S3 privé, chiffré, à accès public bloqué et joignable uniquement en TLS : le code et le Dockerfile de l'Analyzer, le compose, la configuration et le dashboard Grafana, et le jeu de rejeu. Les chemins restent relatifs à la racine du dépôt, ce qui permet au compose de fonctionner tel quel sur la VM. Ni `.env`, ni venv, ni tests, ni certificats ne sont envoyés : un fichier qui ressemble à un secret dans ces dossiers bloque la création de l'archive. La somme SHA-256 de l'archive est enregistrée dans `.state.json`, et la VM la vérifie avant de déployer. Une archive remplacée dans S3 entre-temps est donc refusée.

**Relancer cette cellule, puis l'étape 8, redéploie la version courante du code.**

In [7]:
resources.ensure_bucket(clients.s3, cfg)
data = bundle.build_bundle(ROOT)
sha256 = resources.upload_bundle(clients.s3, cfg, data)
save_state(bucket=cfg.bucket, bundle_key=cfg.bundle_key, bundle_sha256=sha256)
print(f"s3://{cfg.bucket}/{cfg.bundle_key} : {len(data) / 1e6:.1f} Mo, sha256 {sha256[:12]}…")

s3://coffee-vm-352909266144/bundle/app.tar.gz : 2.1 Mo, sha256 659a823bbea1…


## 6. Instance EC2

- **Amazon Linux 2023**, dernière AMI de la région ;
- `t3.small` ;
- volume gp3 de 20 Go **chiffré** ;
- **IMDSv2 obligatoire**, limité à 2 sauts réseau pour que le conteneur de l'Analyzer obtienne les droits du rôle (étape 3) ;
- pas de paire de clés.

Au premier démarrage, le *user-data* installe Docker, puis les plugins `docker compose` et `buildx`, à des **versions épinglées** dont la somme SHA-256 publiée est vérifiée.

Une instance déjà créée est réutilisée (et redémarrée si elle est arrêtée). Son security group, son instance profile et ses options IMDS sont remis aux valeurs attendues. Le user-data ne s'exécute qu'à la création.

In [8]:
state = load_state()
instance_id = resources.ensure_instance(
    clients.ec2,
    cfg,
    ami_id=resources.latest_ami(clients.ssm),
    security_group_id=state["security_group_id"],
    instance_profile_arn=state["instance_profile_arn"],
    user_data=scripts.user_data(),
)
save_state(instance_id=instance_id)
print("instance :", instance_id)

instance : i-09a50b3ee6b938a0b


## 7. Attente de la VM

D'abord l'état `running`, puis l'agent SSM `Online`. Pour une nouvelle instance, cela prend 1 à 3 minutes.

In [10]:
instance_id = load_state()["instance_id"]
resources.wait_running(clients.ec2, instance_id)
resources.wait_ssm_online(clients.ssm, instance_id)
print(instance_id, ": running, agent SSM online")

i-09a50b3ee6b938a0b : running, agent SSM online


## 8. Déploiement

Le script ci-dessous est exécuté sur la VM par **SSM Run Command** :
1. attente de la fin du premier démarrage. S'il a échoué, la cellule s'arrête et affiche la fin de `/var/log/cloud-init-output.log` ;
2. téléchargement de l'archive, **vérification de sa somme SHA-256**, puis décompression dans un dossier de préparation ;
3. écriture de `AWS/Grafana/.env`, avec des droits réservés à root, à partir du mot de passe lu dans Parameter Store, de la source de l'Analyzer (`ANALYSER_SOURCE`) et de l'adresse d'écoute de Grafana (`GRAFANA_BIND` : `0.0.0.0` si `GRAFANA_CIDRS` n'est pas vide, `127.0.0.1` sinon) ;
4. synchronisation vers `/opt/coffee/app`. En cas d'échec avant cette étape, l'application en service n'est pas touchée ;
5. `docker compose -f docker-compose.local.yml up -d --build`, puis Grafana est recréé pour recharger sa configuration.

Le premier déploiement construit l'image de l'Analyzer, télécharge Grafana et installe le plugin Infinity : compter quelques minutes.

`ANALYSER_SOURCE = "timestream"` lit la base AWS, `"file"` rejoue `batches.jsonl`. Changer la valeur et relancer la cellule suffit : compose recrée l'Analyzer quand sa configuration change.

`REPLAY_RESTART = True` relance le rejeu depuis le début, utile quand les 8 h de données ont été rejouées et que toutes les machines sont passées hors ligne.

In [11]:
ANALYSER_SOURCE = "timestream"  # ou "file" (rejeu)
REPLAY_RESTART = False

# Configuration optionnelle d'Amazon SNS (si déployé)
sns_topic_arn = ""
sns_config_path = ROOT / "AWS/SNS/sns_config.json"
if sns_config_path.is_file():
    with sns_config_path.open() as f:
        sns_topic_arn = json.load(f).get("topic_arn", "")

state = load_state()
commands = scripts.deploy_commands(
    cfg.bucket, cfg.bundle_key, state["bundle_sha256"], ACCOUNT_ID, cfg.parameter_name, REGION,
    restart_replay=REPLAY_RESTART, analyser_source=ANALYSER_SOURCE, public=cfg.grafana_public,
    sns_topic_arn=sns_topic_arn,
)
output = resources.run_command(clients.ssm, state["instance_id"], commands, timeout_s=1800)
print(output[-3000:] or "déployé")

1 boto3-1.43.119 botocore-1.43.119 click-8.5.0 fastapi-0.151.0 h11-0.16.0 idna-3.20 jmespath-1.1.0 opentelemetry-api-1.45.1 pydantic-2.14.1 pydantic-core-2.50.1 python-dateutil-2.9.0.post0 s3transfer-0.19.3 six-1.17.0 starlette-1.9.0 typing-extensions-4.16.0 typing-inspection-0.4.4 urllib3-2.8.0 uvicorn-0.54.0
#9 7.485 WARNING: Running pip as the 'root' user can result in broken permissions and conflicting behaviour with the system package manager, possibly rendering your system unusable. It is recommended to use a virtual environment instead: https://pip.pypa.io/warnings/venv. Use the --root-user-action option if you know what you are doing and want to suppress this warning.
#9 7.551 
#9 7.551 [notice] A new release of pip is available: 25.0.1 -> 26.2.2
#9 7.551 [notice] To update, run: pip install --upgrade pip
#9 DONE 7.9s

#10 [5/7] COPY pyproject.toml README.md ./
#10 DONE 0.0s

#11 [6/7] COPY src ./src
#11 DONE 0.0s

#12 [7/7] RUN pip install --no-cache-dir --no-deps .     && use

## 9. Vérification et accès à Grafana

On attend que les conteneurs `analyser` et `grafana` soient `healthy` et que Grafana réponde sur la VM. La cellule affiche ensuite l'**URL publique** de Grafana et vérifie qu'elle répond depuis ce poste. Si ce n'est pas le cas, l'IP publique de ce poste a sans doute changé : relancer les étapes 0 et 4.

Se connecter avec l'utilisateur `admin` et le mot de passe de l'étape 2. Le dashboard **Machines à café — flotte** est la page d'accueil.

L'IP publique de la VM **change à chaque arrêt/redémarrage** (étape 10) : relancer cette cellule pour obtenir la nouvelle URL.

Le tunnel SSM reste possible, même avec `GRAFANA_CIDRS = []` : lancer la commande affichée dans un **terminal**, la laisser ouverte, puis ouvrir http://localhost:3000.

In [12]:
instance_id = load_state()["instance_id"]
status = resources.wait_healthy(clients.ssm, instance_id)
print("\nprêt :", status)
if cfg.grafana_public:
    url = f"http://{resources.public_ip(clients.ec2, instance_id)}:{resources.GRAFANA_PORT}"
    print(f"\nGrafana : {url}  (ouvert à {', '.join(cfg.grafana_cidrs)})")
    reachable = resources.grafana_reachable(url)
    print("joignable depuis ce poste :", "oui" if reachable else "NON (relancer les étapes 0 et 4)")
print("\ntunnel SSM (facultatif, dans un terminal) :")
print(f"  aws ssm start-session --region {REGION} --target {instance_id} "
      "--document-name AWS-StartPortForwardingSession --parameters portNumber=3000,localPortNumber=3000")
print("puis http://localhost:3000")

analyser=healthy, grafana=healthy, Grafana /api/health: oui

prêt : analyser=healthy, grafana=healthy, Grafana /api/health: oui

Grafana : http://3.85.30.233:3000  (ouvert à 0.0.0.0/0)
joignable depuis ce poste : oui

tunnel SSM (facultatif, dans un terminal) :
  aws ssm start-session --region us-east-1 --target i-09a50b3ee6b938a0b --document-name AWS-StartPortForwardingSession --parameters portNumber=3000,localPortNumber=3000
puis http://localhost:3000


## 10. Arrêter / redémarrer la VM

Allumée en continu, la VM coûte environ 20 USD/mois (environ 15 USD pour l'instance, 3,6 USD pour son IPv4 publique et 1,6 USD pour le disque). Arrêtée, elle ne coûte plus que son disque. Au redémarrage, Docker relance les conteneurs (`restart: unless-stopped`), et le rejeu repart du début. La VM reçoit alors une **nouvelle IP publique** : relancer ensuite les étapes 7 et 9, qui affiche la nouvelle URL.

In [ ]:
STOP = True
START = False

instance_id = load_state()["instance_id"]
if STOP:
    resources.set_running(clients.ec2, instance_id, running=False)
    print("arrêt demandé :", instance_id)
elif START:
    resources.set_running(clients.ec2, instance_id, running=True)
    print("démarrage demandé :", instance_id, "(relancer les étapes 7 et 9)")
else:
    print("STOP et START à False : rien à faire")

## 11. Nettoyage (optionnel)

Supprime **tout** : l'instance et son disque, le security group, le rôle et l'instance profile, le bucket et l'archive, le paramètre du mot de passe, puis `.state.json`. Mettre `CLEANUP = True` pour l'exécuter.

In [ ]:
CLEANUP = False

if CLEANUP:
    removed = resources.cleanup(clients, cfg)
    STATE_FILE.unlink(missing_ok=True)
    print("supprimé :", *removed, sep="\n  ") if removed else print("rien à supprimer")
else:
    print("CLEANUP = False : rien n'est supprimé")